In [14]:
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import re

In [15]:
def parse_multi_report_pubtator(raw_data):
    # If the input is bytes, decode it to a string first
    if isinstance(raw_data, bytes):
        raw_data = raw_data.decode('utf-8')

    report_groups = defaultdict(lambda: {'title': '', 'abstract': '', 'annotations': []})
    lines = raw_data.strip().split('\n')
    
    # 1. Grouping Phase: Separate data by PMID
    for line in lines:
        if not line.strip(): 
            continue
        
        parts = line.split('|')
        if len(parts) > 2:
            pmid, type_flag, content = parts[0], parts[1], parts[2]
            if type_flag == 't':
                report_groups[pmid]['title'] = content
            elif type_flag == 'a':
                report_groups[pmid]['abstract'] = content
        else:
            tab_parts = line.split('\t')
            if len(tab_parts) >= 5:
                pmid = tab_parts[0]
                report_groups[pmid]['annotations'].append({
                    'start': int(tab_parts[1]),
                    'end': int(tab_parts[2]),
                    'tags': tab_parts[4].split(',')
                })

    all_results = []

    # 2. Processing Phase: Handle each report independently
    for pmid, data in report_groups.items():
        # Using a space to separate title and abstract to maintain offset integrity
        full_text = data['title'] + " " + data['abstract']
        
        report_words = []
        report_tags = []
        
        for match in re.finditer(r'\w+', full_text):
            start, end = match.start(), match.end()
            word = match.group()
            
            current_word_tags = set()
            for ann in data['annotations']:
                if start >= ann['start'] and end <= ann['end']:
                    for t in ann['tags']:
                        current_word_tags.add(t)
            
            report_words.append(word)
            
            # --- Logic Update Start ---
            # If the set is empty, we tag it with 'O'
            if not current_word_tags:
                report_tags.append(['O'])
            else:
                report_tags.append(sorted(list(current_word_tags)))
            # --- Logic Update End ---
        
        all_results.append({
            'pmid': pmid,
            'words': report_words,
            'tags': report_tags
        })
        
    return all_results

In [16]:
with open("metadata/corpus_pubtator.txt", "r") as f:
    full_text = f.read()

results = parse_multi_report_pubtator(full_text)

In [17]:
def get_tag_df(processed_reports):
    rows = []
    for report in processed_reports:
        for word, tags in zip(report['words'], report['tags']):
            for tag in tags:
                rows.append({
                    'pmid': report['pmid'],
                    'word': word,
                    'tag': tag
                })
    return pd.DataFrame(rows)

# Generate the DataFrame
df = get_tag_df(results)

In [21]:
df_entities = df[df['tag'] != 'O'].reset_index(drop=True)

In [22]:
df_entities

,pmid,word,tag
0,25763772,DCTN4,T116
1,25763772,DCTN4,T123
2,25763772,chronic,T047
3,25763772,Pseudomonas,T047
4,25763772,aeruginosa,T047
...,...,...,...
557942,28550521,highest,T080
557943,28550521,accuracy,T080
557944,28550521,interdental,T082
557945,28550521,bone,T201


In [23]:
# 1. Calculate absolute counts
tag_freq = df_entities['tag'].value_counts().reset_index()
tag_freq.columns = ['Medical_Tag', 'Count']

# 2. Calculate percentages to see the relative distribution
tag_freq['Percentage (%)'] = (tag_freq['Count'] / tag_freq['Count'].sum() * 100).round(2)

# 3. Sort by count (value_counts does this by default, but reset_index might change it)
tag_freq = tag_freq.sort_values(by='Count', ascending=False)

print("Tag Frequency Table (Medical Entities Only):")
print(tag_freq.to_string(index=False))

Tag Frequency Table (Medical Entities Only):
Medical_Tag  Count  Percentage (%)
       T080  36879            6.61
       T081  27436            4.92
       T169  26268            4.71
       T033  25114            4.50
       T116  24558            4.40
       T047  19257            3.45
       T061  18340            3.29
       T170  17943            3.22
       T121  16493            2.96
       T123  15786            2.83
       T109  14700            2.63
       T062  14199            2.54
       T079  12626            2.26
       T078  10952            1.96
       T059  10564            1.89
       T025   9486            1.70
       T082   9466            1.70
       T023   9189            1.65
       T058   8790            1.58
       T191   8271            1.48
       T098   7781            1.39
       T052   7733            1.39
       T060   7708            1.38
       T028   6999            1.25
       T101   6758            1.21
       T126   5964            1.07
       T04

In [26]:
tag_freq["Count"].describe()

count      127.000000
mean      4393.283465
std       6441.738539
min          2.000000
25%        670.000000
50%       2005.000000
75%       4756.000000
max      36879.000000
Name: Count, dtype: float64